# PCA Assignments — Summary Notes

Personal reference notebook covering all 9 PCA assignments.  
Each section captures what the assignment covered, the steps taken, and the core concept worth remembering.

---
| Assignment | Topic |
|---|---|
| 1 | Geometry of PCA — Projection & Variance |
| 2 | Eigenvalues = Variance |
| 3 | Full PCA Pipeline from Scratch |
| 4 | PCA as Coordinate Rotation |
| 5 | Reconstruction Error & Information Loss |
| 6 | Effect of Scaling on PCA |
| 7 | Explained Variance Ratio (Manual vs sklearn) |
| 8 | PCA on 3D Data |
| 9 | Conceptual Short Answers |

---
## Assignment 1 — Geometry of PCA: Projection & Variance

**What it covered:**  
Generated 2D correlated data, manually chose a direction vector, projected the data onto it, and compared variance before and after projection against the optimal PCA direction.

**Steps done:**
- Generated 200 samples from a correlated 2D Gaussian
- Projected data onto an arbitrary direction (θ = 70°)
- Measured variance retained: ~68.71%
- Showed that PC1 (first eigenvector) is the direction that retains maximum variance

**Key Insight:**
> PCA finds the eigenvectors of the covariance matrix. The first eigenvector (PC1) points in the direction of **maximum variance** — projecting onto it preserves more information than any other direction.

| Concept | Meaning |
|---|---|
| PC1 | Direction of greatest spread in the data |
| PC2 | Orthogonal direction — second greatest spread |
| Eigenvalue | Amount of variance captured by each PC |


---
## Assignment 2 — Eigenvalues = Variance

**What it covered:**  
Proved the direct link between eigenvalues and variance. For any eigenvector direction, the variance of the projected data equals its eigenvalue — verified numerically.

**Steps done:**
- Generated 300-sample correlated 2D dataset
- Computed covariance matrix manually
- Extracted eigenvalues and eigenvectors via `np.linalg.eig`
- Projected data onto each eigenvector and measured variance
- Verified: `Var(projection onto PC_i) ≈ Lambda_i`
- Plotted eigenvector arrows on the data scatter

**Key Insight:**
> An eigenvalue is not just a number — it literally equals the variance of the data when projected onto that eigenvector's direction. The sum of all eigenvalues equals the total variance (trace of the covariance matrix).

| Fact | Formula |
|---|---|
| Variance along PC_i | = Lambda_i |
| Total variance | = Sum of all Lambda = trace(C) |


---
## Assignment 3 — Full PCA Pipeline from Scratch

**What it covered:**  
Built a complete PCA pipeline on real manufacturing sensor data (100,000 rows, 9 features) using only NumPy — no sklearn allowed.

**Steps done:**

| Step | What was done |
|---|---|
| 1. Standardize | Z-score: `(x - mean) / std` |
| 2. Covariance matrix | `C = X^T @ X / (n-1)`, shape 9×9 |
| 3. Eigendecomposition | `np.linalg.eig(C)` → 9 eigenvalues + eigenvectors |
| 4. Sort | Ordered by descending eigenvalue |
| 5. Select k | Fixed k = 8 components |
| 6. Project | `X_pca = X @ W` → reduces 9D → 8D |

**Key Insight:**
> PCA is just a series of well-defined matrix operations. Standardize → Covariance → Eigen-decompose → Sort → Slice → Project. Each step has a clear purpose, and understanding them manually makes the sklearn black box much less mysterious.


---
## Assignment 4 — PCA as Coordinate Rotation

**What it covered:**  
Visualized PCA geometrically — showing that PCA does not move the data, it rotates the coordinate system so the axes align with the data's natural directions of spread.

**Steps done:**
- Generated correlated 2D data
- Extracted PC1 and PC2 via manual eigendecomposition
- Plotted original axes (gray dashed) and PCA axes (eigenvector arrows) on the same scatter
- Projected data into PCA space and showed the blob becomes axis-aligned
- Verified off-diagonal covariance drops to ~0 after rotation

**Key Insight:**
> PCA is a rotation. The data cloud does not change shape — only the ruler used to measure it changes. In the rotated frame, features become uncorrelated (covariance matrix goes diagonal).

| Space | Data shape | Covariance matrix |
|---|---|---|
| Original | Tilted ellipse — correlated | Off-diagonal ≠ 0 |
| PCA (rotated) | Axis-aligned ellipse — decorrelated | Off-diagonal ≈ 0 |

---
## Assignment 5 — Reconstruction Error & Information Loss

**What it covered:**  
Reduced 2D data to 1D by keeping only PC1, then reconstructed back to 2D — and measured exactly how much information was lost in the process.

**Steps done:**
- Generated 200-sample correlated 2D dataset
- Projected onto PC1 only → 1D scores
- Reconstructed: `X_reconstructed = scores × PC1 + mean`
- Every reconstructed point lies on the PC1 line
- Measured per-point Euclidean error and verified: `Mean Squared Error ≈ Lambda_2`
- Plotted original vs reconstructed with gray error lines

**Key Insight:**
> What is lost is the PC2 component of each point — its distance off the PC1 line. The reconstruction error equals Lambda_2 (the dropped eigenvalue). PCA always drops the dimension with the least variance, so the loss is minimized compared to dropping any other direction.

| Quantity | Formula |
|---|---|
| Variance retained | Lambda_1 / (Lambda_1 + Lambda_2) × 100% |
| Variance lost | Lambda_2 / (Lambda_1 + Lambda_2) × 100% |
| Mean squared error | ≈ Lambda_2 |


---
## Assignment 6 — Effect of Scaling on PCA

**What it covered:**  
Demonstrated what happens when features have very different scales — and why standardization is non-negotiable before PCA.

**Steps done:**
- Created a 3-feature dataset: Temperature (~tens), Pressure (~thousands), Vibration (~fractions) — all driven by the same underlying signal
- Ran PCA without scaling → PC1 loading: ~1.0 on Pressure, ~0.0 on others
- Ran PCA with StandardScaler → PC1 loaded all three features equally (~0.577 each)
- Plotted side-by-side bar charts and scree curves for comparison

**Key Insight:**
> Without scaling, PCA measures variance in raw units. A feature measured in thousands dominates simply because of its size, not its information content. Standardization puts all features on equal footing.

| Situation | Scale first? |
|---|---|
| Features have different units | **Always scale** |
| Features already on the same scale | Optional |
| Larger-variance features should dominate | Skip scaling |

---
## Assignment 7 — Explained Variance Ratio (Manual vs sklearn)

**What it covered:**  
Computed the Explained Variance Ratio (EVR) manually using eigenvalues and verified it matches sklearn's `explained_variance_ratio_` exactly.

**Steps done:**
- Loaded manufacturing dataset (100k rows, 9 features) using **pandas**
- Standardized with pandas `.mean()` / `.std()`
- Computed EVR manually: `Lambda_i / Sum(Lambda)`
- Ran `sklearn.decomposition.PCA` and compared outputs
- All 9 PC values matched (tolerance 1e-4)
- Visualized with side-by-side bars and cumulative scree curves

**Key Insight:**
> EVR tells you what fraction of total information each PC captures. After standardization, `Sum(Lambda) = number of features` — so a Lambda > 1 means that PC captures more variance than a single original feature would.

| Concept | Formula | Meaning |
|---|---|---|
| Lambda_i | — | Variance along PC_i |
| EVR | `Lambda_i / Sum(Lambda)` | Fraction of total variance |
| Cumulative EVR | Running sum of EVR | Total variance in top-k PCs |


---
## Assignment 8 — PCA on 3D Data

**What it covered:**  
Extended PCA from 2D to 3D — generated correlated 3D data, reduced it to 2D using sklearn, and visualized both the original 3D cloud and the 2D projection.

**Steps done:**
- Created 300-sample correlated 3D Gaussian dataset stored in a **pandas** DataFrame
- Standardized with `sklearn StandardScaler`
- Applied `sklearn PCA(n_components=2)` — reduced 3D → 2D
- Built summary tables (Lambda, EVR, cumulative%) and loadings table using pandas
- Plotted 3D scatter (original space, colored by PC1 score)
- Plotted 2D scatter (PCA space) + bar chart showing variance retained vs lost

**Key Insight:**
> A correlated 3D cloud is not truly 3D at heart — its variance is concentrated along 1–2 directions. PCA finds those directions and collapses the third dimension, retaining most of the structure in just 2D. The same logic scales to any number of dimensions.

| Tool | Used for |
|---|---|
| pandas | Data storage, `.describe()`, `.corr()`, results tables |
| sklearn StandardScaler | Standardization |
| sklearn PCA | Dimensionality reduction |
| Matplotlib 3D | Visualizing original space |
